# 05 — Missing-Modality Robustness Evaluation

## Objective

Evaluate whether training with modality dropout improves robustness when one input modality is unavailable.

**Implemented modalities:** ECG signal and age/sex metadata. This experiment does not use an independent clinical text modality.

## Experimental setup

- Dataset: PTB-XL
- Source training site: site 1
- Target evaluation site: site 2
- Evaluation conditions: full fusion, signal-only, metadata-only
- Methods: baseline fusion without modality dropout and robust fusion with modality dropout
- Random seeds: 21, 42, 123
- Primary metric: macro-F1
- Rare-class metric: HYP F1

The comparison uses the same target evaluation setup for both methods. The three random seeds measure sensitivity to initialization and training randomness, not independent external validation.

## Results


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/Sakina152/mm-dg-healthcare.git"
REPO_DIR = "/content/mm-dg-healthcare"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!git log -1 --oneline

Mounted at /content/drive
Working directory: /content/mm-dg-healthcare
ec7c54c (HEAD -> main, origin/main, origin/HEAD) Document missing-modality robustness evaluation.


In [3]:
from pathlib import Path
import pandas as pd

results_root = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/results_backup"
)
path = results_root / "modality_degradation_comparison.csv"

assert path.is_file(), (
    f"Missing {path}. Run the degradation analysis in Notebook 05 first."
)

degradation = pd.read_csv(path)

print("Per-seed degradation: negative values indicate a performance drop.")
display(
    degradation.sort_values(
        ["metric", "condition", "method", "seed"]
    ).round(4)
)

summary = (
    degradation.groupby(["method", "metric", "condition"])["change_vs_full"]
    .agg(["mean", "std"])
    .reset_index()
    .round(4)
)

print("Mean and standard deviation of degradation across seeds:")
display(summary)


Per-seed degradation: negative values indicate a performance drop.


,method,seed,metric,condition,full_score,missing_condition_score,change_vs_full
3,Baseline,21,HYP_f1,metadata_only,0.3660,0.0000,-0.3660
7,Baseline,42,HYP_f1,metadata_only,0.4168,0.0000,-0.4168
11,Baseline,123,HYP_f1,metadata_only,0.3702,0.0000,-0.3702
15,Robust,21,HYP_f1,metadata_only,0.4257,0.0000,-0.4257
19,Robust,42,HYP_f1,metadata_only,0.3947,0.0000,-0.3947
23,Robust,123,HYP_f1,metadata_only,0.4249,0.0000,-0.4249
2,Baseline,21,HYP_f1,signal_only,0.3660,0.3548,-0.0112
6,Baseline,42,HYP_f1,signal_only,0.4168,0.4111,-0.0057
10,Baseline,123,HYP_f1,signal_only,0.3702,0.3792,0.0090
14,Robust,21,HYP_f1,signal_only,0.4257,0.4236,-0.0021


Mean and standard deviation of degradation across seeds:


,method,metric,condition,mean,std
0,Baseline,HYP_f1,metadata_only,-0.3844,0.0282
1,Baseline,HYP_f1,signal_only,-0.0027,0.0104
2,Baseline,macro_f1,metadata_only,-0.3329,0.0831
3,Baseline,macro_f1,signal_only,-0.0043,0.0087
4,Robust,HYP_f1,metadata_only,-0.4151,0.0177
5,Robust,HYP_f1,signal_only,-0.0003,0.0029
6,Robust,macro_f1,metadata_only,-0.4158,0.0309
7,Robust,macro_f1,signal_only,-0.0020,0.0016


## Interpretation

### 1. Missing metadata

When metadata is unavailable, the model is evaluated using the ECG signal alone. The mean macro-F1 degradation was approximately **0.0043** for baseline fusion and **0.0020** for robust fusion. Mean HYP F1 degradation was approximately **0.0027** and **0.0003**, respectively.

These results suggest that modality-dropout training may improve stability when metadata is missing.

### 2. Missing ECG signal

When the ECG signal is unavailable, the model is evaluated using age/sex metadata alone. Both approaches show a large drop in macro-F1. The mean drop was approximately **0.3329** for baseline fusion and **0.4158** for robust fusion.

Therefore, the current experiment does not demonstrate improved robustness when ECG is missing. The metadata-only model also achieved HYP F1 of zero in all six runs.

### 3. Limitations

- Only three random seeds were evaluated.
- All runs use the same target split; the seeds are not independent external validation cohorts.
- Results do not establish statistical significance or clinical reliability.
- The metadata-only condition has limited predictive performance, particularly for HYP.
- The implemented modalities are ECG and age/sex metadata, not ECG and independent clinical text.

## Conclusion

The results support a preliminary finding that modality dropout improves stability when metadata is missing. However, robustness to missing ECG remains unresolved. Further work should investigate this limitation and evaluate per-class performance before making stronger claims.
